# 02.2 CNN Basics

This notebook is the true entry point into Convolutional Neural Networks (CNNs).

Key concepts:

- convolution layer
- kernel
- stride
- padding
- pooling
- feature map
- channel changes

## Learning Goals

After this notebook, you should be able to:

1. Understand the input-output structure of `Conv2d`.
2. Explain how shapes change after convolution.
3. Understand the roles of `stride` and `padding`.
4. Understand pooling layers.
5. Read the shape flow of a small CNN.
6. Prepare structurally for later image classification.

In [ ]:
import torch
import torch.nn as nn

## Inputs and Outputs of `Conv2d`

The input to `Conv2d` is usually:

- `(N, C, H, W)`

where:

- `N`: batch size
- number of input channels
- height
- width

The output is usually:

- `(N, C_out, H_out, W_out)`

In [ ]:
x = torch.randn(4, 1, 8, 8)
conv = nn.Conv2d(in_channels=1, out_channels=3, kernel_size=3, stride=1, padding=1)
y = conv(x)

print("x.shape =", x.shape)
print("y.shape =", y.shape)

The output shape here is `(4, 3, 8, 8)` because:

- batch size stays the same
- output channels are determined by `out_channels=3`
- with `padding=1` and `stride=1`, spatial size stays unchanged

## 2. `kernel_size`、`stride`、`padding`
## `kernel_size`, `stride`, and `padding`

These three arguments determine the spatial size of the output feature map.

First build intuition:

- larger receptive patch each step
- larger jumps, smaller output
- preserves more border information

In [ ]:
x = torch.randn(1, 1, 8, 8)

conv_a = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=0)
conv_b = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
conv_c = nn.Conv2d(1, 4, kernel_size=3, stride=2, padding=1)

print("input shape =", x.shape)
print("padding=0, stride=1 ->", conv_a(x).shape)
print("padding=1, stride=1 ->", conv_b(x).shape)
print("padding=1, stride=2 ->", conv_c(x).shape)

## Number of Parameters

Convolution layers also contain learnable parameters.

For `Conv2d(in_channels=C_in, out_channels=C_out, kernel_size=k)`:

- number of weight parameters: `C_out * C_in * k * k`
- add another `C_out` if bias is used

In [ ]:
conv = nn.Conv2d(1, 3, kernel_size=3)

num_params = sum(p.numel() for p in conv.parameters())
print("num_params =", num_params)

# weights = 3 * 1 * 3 * 3 = 27
# bias = 3
# total = 30

In [ ]:
# Exercise 1
# Conv2d(in_channels=3, out_channels=8, kernel_size=5)
#Manually compute the total parameter count, then verify it with code.
# Manually compute the total number of parameters, then verify with code.

# conv_ex =
# print(sum(p.numel() for p in conv_ex.parameters()))

In [ ]:
# Exercise 1 Reference Solution

conv_ex = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=5)
print(sum(p.numel() for p in conv_ex.parameters()))

# weights = 8 * 3 * 5 * 5 = 600
# bias = 8
# total = 608

## Pooling Layers

Pooling is commonly used to reduce spatial dimensions.

The most common one is:

- `MaxPool2d`

Its intuition is: take the maximum value within a small local window.


In [ ]:
x = torch.randn(2, 4, 8, 8)
pool = nn.MaxPool2d(kernel_size=2, stride=2)
y = pool(x)

print("x.shape =", x.shape)
print("y.shape =", y.shape)

Here the spatial size changes from `8x8` to `4x4` because a `2x2` pooling window with `stride=2` halves both height and width.


## Shape Flow in a Minimal CNN

When reading CNN code, the most important thing is to keep tracking the shape.


In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)
        self.fc = nn.Linear(16 * 2 * 2, 10)

    def forward(self, x):
        print("input:", x.shape)
        x = self.conv1(x)
        print("after conv1:", x.shape)
        x = self.relu1(x)
        x = self.pool1(x)
        print("after pool1:", x.shape)
        x = self.conv2(x)
        print("after conv2:", x.shape)
        x = self.relu2(x)
        x = self.pool2(x)
        print("after pool2:", x.shape)
        x = x.flatten(start_dim=1)
        print("after flatten:", x.shape)
        x = self.fc(x)
        print("after fc:", x.shape)
        return x


model = TinyCNN()
dummy = torch.randn(4, 1, 8, 8)
out = model(dummy)

Pay special attention to this shape chain:

- `(4, 1, 8, 8)`
- `(4, 8, 8, 8)`
- `(4, 8, 4, 4)`
- `(4, 16, 4, 4)`
- `(4, 16, 2, 2)`
- `(4, 64)`
- `(4, 10)`

This is the basic skill required to read CNN code.


In [ ]:
# Exercise 2
# If the input is (batch, 1, 8, 8), and it passes through:
# 1. Conv2d(1, 4, kernel_size=3, padding=1)
# 2. MaxPool2d(2)
# What is the output shape?
#Compute it manually first, then verify it with code.
# Compute it by hand first, then verify with code.

# x =
# conv =
# pool =
# y =
# print(y.shape)

In [ ]:
# Exercise 2 Reference Solution

x = torch.randn(5, 1, 8, 8)
conv = nn.Conv2d(1, 4, kernel_size=3, padding=1)
pool = nn.MaxPool2d(2)
y = pool(conv(x))
print(y.shape)

# shape: (5, 4, 4, 4)

## `Flatten` and Linear Layers

The output of convolution layers is still usually a 4D tensor.

If a linear layer comes next, we usually flatten first.


In [ ]:
x = torch.randn(3, 16, 2, 2)
x_flat = x.flatten(start_dim=1)

print("x.shape =", x.shape)
print("x_flat.shape =", x_flat.shape)

`16 * 2 * 2 = 64`, so after flattening the shape becomes `(batch_size, 64)`.


In [ ]:
# Exercise 3
# If a layer output has shape (7, 12, 3, 3),
# what is the shape after flatten(start_dim=1)?


Exercise 3 Reference Answer

`12 * 3 * 3 = 108`, so the shape becomes `(7, 108)`.

## Summary

The most important outcome of this notebook is building CNN shape intuition.

You should now be able to answer:

1. What do the input and output shapes of `Conv2d` usually look like?
2. Why does `out_channels` change the number of feature-map channels?
3. Why do pooling layers often reduce height and width?
4. Why do we often flatten convolution outputs before linear layers?

Suggested next step:

- Move to the CNN classification notebook and put these structures into a real classification task.